# SHIPP — 92 Release Validation

**Purpose:** final read-only validation after Tasks 90 and 91.

This task validates the existing canonical outputs and appends evidence to `bootcamp_students.shipp_gold.release_validation_log`.

It does **not** rebuild Silver, Gold, RAG, Requests, Categories, or App authentication.

Requests and categories are already persisted by Stage 11 (`silver_requests`). This task records their current count and distinct categories as release evidence.

Security: do not store passwords, tokens, API keys, client secrets, or database credentials in this log.


In [0]:
%run ../common/shipp_silver_lib


In [0]:
import importlib
import json
import uuid
from datetime import datetime, timezone

release_validation = importlib.reload(
    importlib.import_module("data_pipeline.quality.release_validation")
)

TABLE_MAP = {
    "bronze_route_responses": BRONZE_ROUTE_RESPONSES,
    "bronze_vision_responses": BRONZE_VISION_RESPONSES,
    "silver_listings": SILVER_LISTINGS,
    "silver_requests": SILVER_REQUESTS,
    "silver_routes": SILVER_ROUTES,
    "silver_listing_files": SILVER_LISTING_FILES,
    "silver_listing_content": SILVER_LISTING_CONTENT,
    "silver_semantic_scores": SILVER_SEMANTIC_SCORES,
    "gold_candidate_matches": GOLD_CANDIDATE_MATCHES,
    "gold_listing_search_docs": GOLD_LISTING_SEARCH_DOCS,
    "gold_marketplace_metrics": GOLD_MARKETPLACE_METRICS,
    "rag_eval_results": RAG_EVAL_RESULTS,
    "pipeline_run_log": PIPELINE_RUN_LOG,
}

VALIDATION_LOG = f"{CATALOG}.{GOLD_SCHEMA}.release_validation_log"
VALIDATION_RUN_ID = str(uuid.uuid4())

print("VALIDATION_RUN_ID =", VALIDATION_RUN_ID)
print("VALIDATION_LOG    =", VALIDATION_LOG)


In [0]:
summary, checks = release_validation.run_release_validation(spark, TABLE_MAP)

check_rows = [
    {
        "check_name": c.check_name,
        "status": c.status,
        "observed_value": c.observed_value,
        "detail": c.detail,
    }
    for c in checks
]

print(json.dumps(summary, indent=2, default=str))
display(spark.createDataFrame(check_rows).orderBy("status", "check_name"))


In [0]:
# Persist evidence only. Existing business products are untouched.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {VALIDATION_LOG} (
    validation_run_id STRING,
    checked_at TIMESTAMP,
    overall_status STRING,
    check_name STRING,
    check_status STRING,
    observed_value STRING,
    detail STRING,
    request_count BIGINT,
    request_categories STRING,
    listing_categories STRING,
    gold_match_count BIGINT,
    search_doc_count BIGINT
) USING DELTA
""")

checked_at = datetime.now(timezone.utc).replace(tzinfo=None)
request_categories_json = json.dumps(summary.get("request_categories", []))
listing_categories_json = json.dumps(summary.get("listing_categories", []))

rows = [
    (
        VALIDATION_RUN_ID,
        checked_at,
        summary["status"],
        c.check_name,
        c.status,
        c.observed_value,
        c.detail,
        int(summary.get("request_count", 0)),
        request_categories_json,
        listing_categories_json,
        int(summary.get("gold_match_count", 0)),
        int(summary.get("search_doc_count", 0)),
    )
    for c in checks
]

LOG_SCHEMA = (
    "validation_run_id string, checked_at timestamp, overall_status string, "
    "check_name string, check_status string, observed_value string, detail string, "
    "request_count long, request_categories string, listing_categories string, "
    "gold_match_count long, search_doc_count long"
)

(
    spark.createDataFrame(rows, LOG_SCHEMA)
    .write.format("delta")
    .mode("append")
    .saveAsTable(VALIDATION_LOG)
)

display(
    spark.table(VALIDATION_LOG)
    .filter(F.col("validation_run_id") == VALIDATION_RUN_ID)
    .orderBy("check_status", "check_name")
)

print("Saved request count       :", summary.get("request_count"))
print("Saved request categories  :", summary.get("request_categories"))
print("Saved listing categories  :", summary.get("listing_categories"))


In [0]:
result = {
    "validation_run_id": VALIDATION_RUN_ID,
    "log_table": VALIDATION_LOG,
    **summary,
}

if summary["status"] != "PASS":
    raise AssertionError(
        "RELEASE VALIDATION FAILED: " + ", ".join(summary["failed_checks"])
    )

print("PASS — SHIPP release validation completed.")
dbutils.notebook.exit(json.dumps(result, default=str))
